# Qdrant + Neo4j 하이브리드 RAG 평가 파이프라인 (RAGAS 미사용, 자체 구현)

이 노트북은 다음 순서로 동작한다.

1. **코퍼스 로드** — 챗봇이 실제로 검색하는 두 소스(Qdrant 벡터 청크 + Neo4j 조문)에서 무작위 샘플을 가져온다.
2. **테스트셋 생성** — RAGAS 없이, `ChatOpenAI`를 직접 호출해 문서 하나당 한국어 질문/정답 쌍을 생성한다.
3. **챗봇 실행** — `LangGraphChatbot`으로 각 질문에 답변하고, 검색된 컨텍스트를 함께 수집한다.
4. **평가** — RAGAS 없이, `ChatOpenAI`를 LLM 심사위원으로 사용해 faithfulness / answer_relevancy / context_precision / context_recall 4개 지표를 직접 계산한다. 프롬프트와 출력이 모두 한국어이므로 결과도 한국어로 나온다.

> **설계 전제 (확인 완료)**
> - 챗봇은 Qdrant와 Neo4j **둘 다** 검색하는 하이브리드 구조이므로, 테스트셋 코퍼스도 두 소스를 모두 포함한다.
> - Qdrant와 Neo4j는 **서로 다른 별개의 데이터**이므로 중복 문제는 없으며, `metadata["source"]`로 출처를 구분한다.
> - 도메인은 **군 복지·혜택**과 **법령·조문**을 모두 다루므로, 질문 생성 페르소나를 두 갈래로 구성한다.
> - RAGAS는 내부 프롬프트가 영어라 언어 적응(adapt_prompts)을 거쳐도 한국어 출력이 불안정했다. 이 노트북은 RAGAS를 완전히 걷어내고, 테스트셋 생성과 평가를 모두 직접 작성한 한국어 프롬프트 + LLM 호출로 대체한다.

필요 패키지:
```
pip install langchain-openai langchain-qdrant qdrant-client neo4j pandas python-dotenv
```


In [1]:
# =====================================================================
# 임포트 및 환경 변수 로드 (RAGAS 미사용)
# =====================================================================
import os
import sys
import json
import re
import random
from dataclasses import dataclass
from typing import List, Optional

import pandas as pd
from dotenv import load_dotenv

from qdrant_client import QdrantClient
from neo4j import GraphDatabase
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI

# .env 는 프로그램 시작 시점에 한 번만 로드한다.
load_dotenv()


c:\3번쨰 플젝\SKN31-3rd-2Team\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## 0. 설정

수정이 필요한 값은 이 셀 하나에만 모아 둔다. (기존 코드에서 여러 셀에 중복 정의되던 설정을 통합했다.)

In [2]:
# =====================================================================
# 전역 설정 — 값 수정은 이 셀에서만 한다.
# =====================================================================
# Qdrant
QDRANT_URL      = "http://localhost:6333"
COLLECTION_NAME = "guidance_vectordb"

# Neo4j (미지정 시 .env 값 사용)
NEO4J_DATABASE  = os.getenv("NEO4J_DATABASE", "lawdb")

# 코퍼스 샘플링
TOTAL_CHUNKS    = 100   # 두 소스에서 합쳐 가져올 총 문서 수
TESTSET_SIZE    = 20    # 생성할 질문/정답 쌍 개수

# 모델
GEN_LLM_MODEL   = "gpt-4o"          # 테스트셋 생성용
EVAL_LLM_MODEL  = "gpt-4o-mini"     # 평가용 (비용 절감)

# 출력 파일
TESTSET_CSV     = "rag_testset.csv"
RESULT_CSV      = "rag_eval_result.csv"


## 1. 코퍼스 로드

챗봇이 실제로 검색하는 두 소스에서 무작위 샘플을 가져온다.

- **Qdrant** — 벡터 컬렉션에 적재된 청크
- **Neo4j** — `ARTICLE` 노드의 조문 본문

두 소스는 별개의 데이터이므로, 각 `Document` 의 `metadata["source"]` 에 출처를 남겨 이후 추적을 쉽게 한다.

In [3]:
def fetch_random_chunks_from_qdrant(
    collection_name: str,
    limit: int = 50,
    url: str = "http://localhost:6333",
) -> list[Document]:
    """Qdrant 컬렉션에서 무작위로 지정 개수만큼 청크를 가져온다."""
    print(f"[Qdrant] '{collection_name}' 에서 {limit}개 랜덤 청크를 수집한다...")
    client = QdrantClient(url=url)

    # 1) 페이로드 없이 가볍게 스크롤하여 전체 포인트 ID 만 빠르게 수집한다.
    all_ids: list = []
    next_offset = None
    while True:
        records, next_offset = client.scroll(
            collection_name=collection_name,
            limit=500,
            with_payload=False,
            with_vectors=False,
            offset=next_offset,
        )
        all_ids.extend(r.id for r in records)
        if next_offset is None:
            break

    total = len(all_ids)
    if total == 0:
        print("[Qdrant] 경고: 컬렉션에 데이터가 없다.")
        return []

    # 2) 중복 없이 랜덤 ID 추출
    target = min(limit, total)
    random_ids = random.sample(all_ids, target)
    print(f"[Qdrant] 전체 {total}개 중 {target}개를 선정했다.")

    # 3) 선정된 ID 의 페이로드만 일괄 조회
    records = client.retrieve(
        collection_name=collection_name,
        ids=random_ids,
        with_payload=True,
        with_vectors=False,
    )

    # 4) LangChain Document 로 변환
    docs: list[Document] = []
    for rec in records:
        payload = rec.payload or {}
        page_content = (
            payload.get("page_content")
            or payload.get("content")
            or payload.get("text")
        )
        if not page_content:
            continue
        metadata = dict(payload.get("metadata") or {})
        metadata["source"] = "qdrant"        # 출처 표시
        metadata["point_id"] = rec.id
        docs.append(Document(page_content=page_content, metadata=metadata))

    print(f"[Qdrant] {len(docs)}개 청크를 로드했다.")
    return docs

In [4]:
def fetch_random_articles_from_neo4j(
    limit: int = 50,
    uri: str = None,
    username: str = None,
    password: str = None,
    database: str = None,
) -> list[Document]:
    """Neo4j 에서 무작위로 지정 개수만큼 ARTICLE 노드의 본문을 가져온다."""
    uri      = uri      or os.getenv("NEO4J_URI")
    username = username or os.getenv("NEO4J_USERNAME")
    password = password or os.getenv("NEO4J_PASSWORD")
    database = database or NEO4J_DATABASE

    print(f"[Neo4j] '{database}' 에서 {limit}개 랜덤 조문을 수집한다...")
    driver = GraphDatabase.driver(uri, auth=(username, password))

    docs: list[Document] = []
    try:
        with driver.session(database=database) as session:
            results = session.run(
                """
                MATCH (a:ARTICLE)
                WHERE a.description IS NOT NULL
                RETURN a.description AS description, a.id AS id, a.name AS name
                ORDER BY rand()
                LIMIT $k
                """,
                k=limit,
            )
            for r in results:
                docs.append(
                    Document(
                        page_content=r["description"],
                        metadata={
                            "source": "neo4j",       # 출처 표시
                            "article_id": r.get("id"),
                            "article_name": r.get("name"),
                        },
                    )
                )
    finally:
        driver.close()

    print(f"[Neo4j] {len(docs)}개 조문을 로드했다.")
    return docs

In [5]:
def load_corpus(total: int = TOTAL_CHUNKS) -> list[Document]:
    """두 소스에서 절반씩 가져와 하나의 코퍼스로 합친다.

    챗봇이 Qdrant 와 Neo4j 를 모두 검색하므로, 테스트셋 코퍼스도
    두 소스를 함께 담아야 context_recall / precision 이 공정하게 나온다.
    """
    half = total // 2
    print(f"목표 {total}개 = Qdrant {half} + Neo4j {half}\n")

    qdrant_docs = fetch_random_chunks_from_qdrant(
        collection_name=COLLECTION_NAME, limit=half, url=QDRANT_URL
    )
    neo4j_docs = fetch_random_articles_from_neo4j(limit=half)

    docs = qdrant_docs + neo4j_docs
    print(f"\n병합 완료: 총 {len(docs)}개 "
          f"(Qdrant {len(qdrant_docs)} + Neo4j {len(neo4j_docs)})")
    return docs


docs = load_corpus(TOTAL_CHUNKS)

목표 100개 = Qdrant 50 + Neo4j 50

[Qdrant] 'guidance_vectordb' 에서 50개 랜덤 청크를 수집한다...
[Qdrant] 전체 337개 중 50개를 선정했다.
[Qdrant] 50개 청크를 로드했다.
[Neo4j] 'lawdb' 에서 50개 랜덤 조문을 수집한다...
[Neo4j] 50개 조문을 로드했다.

병합 완료: 총 100개 (Qdrant 50 + Neo4j 50)


## 2. 한국어 테스트셋 생성 (자체 구현, RAGAS 미사용)

RAGAS `TestsetGenerator` 대신, 문서 하나당 `ChatOpenAI`를 한 번씩 직접 호출해서
질문/정답 쌍을 만든다.

1. **한국어 생성** — 시스템 프롬프트 자체를 전부 한국어로 작성하고, "반드시 한국어로, 반드시 이 JSON 형식으로만 응답하라"고 명시한다. 언어 적응(adapt_prompts) 같은 별도 단계가 필요 없다.
2. **문서 단위 생성** — 코퍼스에서 `TESTSET_SIZE`개의 문서를 무작위로 뽑아, 각 문서를 유일한 근거(context)로 삼아 질문 1개 + 정답 1개를 생성한다. 문서가 이미 청크/조문 단위로 잘게 나뉘어 있으므로 추가 분할이 필요 없다.
3. **도메인 페르소나** — 군 복지와 법령 두 페르소나를 번갈아 프롬프트에 넣어, 실제 사용자가 던질 법한 질문을 유도한다.


In [6]:
# 도메인 페르소나 — 군 복지 + 법령 두 갈래 (문서마다 번갈아 사용한다)
PERSONAS = [
    {
        "name": "군 복무자",
        "description": (
            "군 복무 중이거나 입대를 앞둔 사람으로, 급여·휴가·의료·주거 등 "
            "복지와 혜택을 구체적으로 알고 싶어 한다."
        ),
    },
    {
        "name": "법령 확인 민원인",
        "description": (
            "관련 법령과 조문의 요건·절차·예외를 정확히 확인하려는 사람으로, "
            "근거 조항과 적용 범위를 중요하게 여긴다."
        ),
    },
]

QA_GEN_SYSTEM_PROMPT = """당신은 한국어 RAG 챗봇 평가용 테스트셋을 만드는 전문가다.
주어진 하나의 문서(context)만 근거로 삼아, 주어진 페르소나가 실제로 물어볼 법한
질문 1개와 그 정답 1개를 생성한다.

규칙:
- 질문과 답변은 반드시 한국어로 작성한다.
- 질문은 반드시 주어진 문서 내용만으로 답할 수 있어야 한다.
- 답변은 문서에 실제로 있는 내용만 근거로 작성하고, 없는 내용을 지어내지 않는다.
- 답변은 2~4문장 정도로 간결하게 작성한다.
- 다른 설명이나 인사말 없이, 반드시 아래 JSON 형식으로만 응답한다.

{"question": "...", "answer": "..."}
"""


def _parse_json_response(text: str) -> dict:
    """LLM 응답에서 코드펜스 등을 제거하고 JSON으로 파싱한다."""
    text = text.strip()
    text = re.sub(r"^```(json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()
    return json.loads(text)


def generate_korean_testset(
    docs: list[Document],
    testset_size: int = TESTSET_SIZE,
) -> pd.DataFrame:
    """RAGAS 없이, LLM 직접 호출로 한국어 질문/정답 테스트셋을 생성한다."""
    llm = ChatOpenAI(model=GEN_LLM_MODEL, temperature=0.7)

    sample_size = min(testset_size, len(docs))
    sampled_docs = random.sample(docs, sample_size)

    rows = []
    for i, doc in enumerate(sampled_docs, start=1):
        persona = PERSONAS[i % len(PERSONAS)]
        user_prompt = (
            f"[페르소나] {persona['name']} - {persona['description']}\n\n"
            f"[문서(context)]\n{doc.page_content}\n\n"
            "위 문서를 근거로 이 페르소나가 물어볼 법한 질문과 정답을 JSON으로 생성해라."
        )
        try:
            resp = llm.invoke(
                [
                    {"role": "system", "content": QA_GEN_SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ]
            )
            parsed = _parse_json_response(resp.content)
            question = str(parsed["question"]).strip()
            answer = str(parsed["answer"]).strip()
        except Exception as e:
            print(f"[{i}/{sample_size}] 생성 실패, 건너뜀: {e}")
            continue

        rows.append(
            {
                "user_input": question,
                "reference": answer,
                "reference_contexts": doc.page_content,
                "source": doc.metadata.get("source", "unknown"),
                "persona": persona["name"],
            }
        )
        print(f"[{i}/{sample_size}] OK  {question[:40]}...")

    testset_df = pd.DataFrame(rows)
    print(f"\n테스트셋 {len(testset_df)}개 생성 완료")
    return testset_df


In [7]:
# 테스트셋 생성 실행
testset_df = generate_korean_testset(docs, testset_size=TESTSET_SIZE)
testset_df.to_csv(TESTSET_CSV, index=False, encoding="utf-8-sig")
print(f"테스트셋 {len(testset_df)}개 저장 완료 -> {TESTSET_CSV}")
testset_df.head()


[1/20] OK  병역법에 따라 병역준비역으로 편입된 사람에게 어떤 절차나 사항을 통지해야...
[2/20] OK  군 복무에 관련된 위원회의 회의는 어떻게 운영되나요?...
[3/20] OK  국군 회원과 기업 회원이 받을 수 있는 혜택의 제공 주기는 어떻게 되나요...
[4/20] OK  전역할 때 전역증은 어떻게 받을 수 있나요?...
[5/20] OK  군 복무 중 성고충 문제에 대한 전문 상담 제도는 어떤 항목에 포함되어 ...
[6/20] OK  2026년에는 초급간부의 기본급이 얼마나 인상되나요?...
[7/20] OK  전직지원교육을 위탁할 수 있는 기관은 어떤 기준으로 선정되나요?...
[8/20] OK  군 복무 중 병 복지 혜택에는 어떤 것들이 있나요?...
[9/20] OK  박사과정전문연구요원이 박사학위 취득 유예기간을 신청하려면 어떤 절차를 따...
[10/20] OK  2026년에 소위 계급의 급여는 얼마나 되나요?...
[11/20] OK  징집되거나 소집된 사람의 복무에 관해서는 어떤 법률이 적용되나요?...
[12/20] OK  군 복무 중 발생한 고충사항은 어디에서 심사받을 수 있나요?...
[13/20] OK  "전사자"와 "순직자"의 정의는 어떻게 다른가요?...
[14/20] OK  군 복무 중 다문화적 가치에 대한 교육은 얼마나 자주 받게 되나요?...
[15/20] OK  군인의 헌법상 권리는 어떤 경우에 제한될 수 있나요?...
[16/20] OK  군 복무자의 경우에도 연봉 감액 기준이 적용되나요?...
[17/20] OK  성고충전문상담관 운영에 대한 절차나 요건은 어떻게 되나요?...
[18/20] OK  군 복무 중인 병사로서 병 회원 저축 상품에 가입하려면 어떤 조건이 필요...
[19/20] OK  출산휴가와 연계하여 3개월 이상 육아휴직을 하는 경우, 결원을 보충할 수...
[20/20] OK  군 복무 중인 사람에게 연금 분할이 어떻게 적용되나요?...

테스트셋 20개 생성 완료
테스트셋 20개 저

,user_input,reference,reference_contexts,source,persona
0,병역법에 따라 병역준비역으로 편입된 사람에게 어떤 절차나 사항을 통지해야 하나요?,"병역법에 따르면 지방병무청장은 병역준비역으로 편입된 사람에게 현역, 보충역, 대체역...",① 지방병무청장(병무지청장을 포함한다. 이하 이 조에서 같다)은 병역준비역으로 편입...,neo4j,법령 확인 민원인
1,군 복무에 관련된 위원회의 회의는 어떻게 운영되나요?,"위원회의 회의는 재적위원 과반수의 출석으로 개의하고, 출석위원 3분의 2 이상의 찬...",① 위원장은 회의를 소집하고 그 의장이 된다.\n② 위원회의 회의는 재적위원 과반수...,neo4j,군 복무자
2,국군 회원과 기업 회원이 받을 수 있는 혜택의 제공 주기는 어떻게 되나요?,"국군 회원은 전용 쿠폰을 월 1회 받을 수 있으며, 기업 회원은 쿠폰을 월별로 변동...","이 표는 국군/기업 회원 대상 숙박·예약 혜택을 요약한 것으로 보이며, 각 컬럼에는...",qdrant,법령 확인 민원인
3,전역할 때 전역증은 어떻게 받을 수 있나요?,전역증은 전역 당일 전역신고 후 개인에게 지급됩니다. 만약 나라사랑카드를 보유하고 ...,2026 병 복지 길라잡이 45\n전 역\n현역병의 예비역 편입\n • 시기 / 권...,qdrant,군 복무자
4,군 복무 중 성고충 문제에 대한 전문 상담 제도는 어떤 항목에 포함되어 있나요?,성고충 문제에 대한 전문 상담 제도는 '성고충전문상담관 운영' 항목에 포함되어 있습...,"이 표는 **인사제도/근무 분야**에 관한 목차형 목록으로, 군 인사·복무 관련 제...",qdrant,법령 확인 민원인


## 3. 챗봇 실행

`LangGraphChatbot` 으로 각 질문에 답변하고, 검색된 컨텍스트를 함께 수집한다.
RAGAS의 `SingleTurnSample` 대신, 같은 필드를 가진 자체 `EvalSample` 데이터클래스를 사용한다.

- `bot.ask(question)` 이 `(references, answer)` 순서로 반환한다고 가정한다. **실제 반환 순서가 다르면 아래 언팩 부분을 수정한다.**
- 한 문항에서 오류가 나도 전체 루프가 멈추지 않도록 예외를 개별 처리한다.

> 아래 import 경로(`run_chatbot`)는 실제 프로젝트 구조에 맞게 조정한다.


In [3]:
import sys
import os

# 현재 노트북 파일의 절대 경로를 구한 뒤, 부모 폴더(evaluation)의 부모 폴더(project_root)를 경로에 추가
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from backend.run_chatbot import LangGraphChatbot


@dataclass
class EvalSample:
    """RAGAS SingleTurnSample 대체용 자체 데이터클래스."""
    user_input: str
    response: str
    retrieved_contexts: List[str]
    reference: Optional[str] = None


def _extract_context_text(ref) -> str:
    """references 원소를 문자열로 변환한다. (Document / dict / str / 커스텀 객체 대응)"""
    if isinstance(ref, str):
        return ref
    if isinstance(ref, Document):
        return ref.page_content
    if isinstance(ref, dict):
        return (
            ref.get("page_content")
            or ref.get("content")
            or ref.get("text")
            or str(ref)
        )
    if hasattr(ref, "page_content"):
        return ref.page_content
    return str(ref)


def run_chatbot_on_testset(testset_df: pd.DataFrame) -> list[EvalSample]:
    bot = LangGraphChatbot(verbose=False)
    samples: list[EvalSample] = []
    total = len(testset_df)

    for i, (_, row) in enumerate(testset_df.iterrows(), start=1):
        question = row["user_input"]
        reference = row.get("reference")
        try:
            # 반환 순서가 (references, answer) 가 아니면 이 줄을 수정한다.
            references, answer = bot.ask(question)
            retrieved_contexts = [_extract_context_text(r) for r in references]

            samples.append(
                EvalSample(
                    user_input=question,
                    response=answer,
                    retrieved_contexts=retrieved_contexts,
                    reference=reference,
                )
            )
            print(f"[{i}/{total}] OK  {question[:40]}...")
        except Exception as e:
            print(f"[{i}/{total}] 실패: {e}  (질문: {question[:40]}...)")

    print(f"\n수집 완료: {len(samples)}/{total} 문항")
    return samples


In [4]:
import sys
import types

# 1. 빈 모듈 객체 생성
dummy_vertexai = types.ModuleType("langchain_community.chat_models.vertexai")

# 2. Ragas가 안에서 꺼내 쓰는 ChatVertexAI 클래스를 공백으로 정의
class ChatVertexAI:
    pass

# 3. 생성한 클래스를 더미 모듈에 주입
dummy_vertexai.ChatVertexAI = ChatVertexAI

# 4. 파이썬 시스템 모듈 관리자(sys.modules)에 '이미 설치된 모듈'인 것처럼 사기 치기
sys.modules["langchain_community.chat_models.vertexai"] = dummy_vertexai

# --- 이제 Ragas가 이 모듈을 import 해도 에러 없이 통과합니다 ---

In [5]:
# =====================================================================
# Ragas 평가 (LLM 심사위원 대신 ragas 공식 메트릭 사용)
# =====================================================================
from ragas import EvaluationDataset, evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import pandas as pd
from ragas.run_config import RunConfig
from ragas import evaluate

testset_df = pd.read_csv('rag_testset.csv')



def run_ragas_evaluation(samples: list[EvalSample]) -> pd.DataFrame:
    """ragas 공식 메트릭(faithfulness, answer_relevancy, context_precision, context_recall)으로 평가한다."""

    # ragas가 내부적으로 쓰는 judge LLM / 임베딩
    evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model=EVAL_LLM_MODEL, temperature=0))
    evaluator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())  # answer_relevancy가 임베딩 필요

    # samples -> ragas가 요구하는 dict 리스트로 변환
    data = [
        {
            "user_input": s.user_input,
            "response": s.response,
            "retrieved_contexts": s.retrieved_contexts,
            "reference": s.reference,
        }
        for s in samples
    ]
    dataset = EvaluationDataset.from_list(data)

    metrics = [
        Faithfulness(),
        AnswerRelevancy(),
        ContextPrecision(),
        ContextRecall(),
    ]

    result = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=evaluator_llm,
        embeddings=evaluator_embeddings
    )

    return result.to_pandas()


# ---------------------------------------------------------------------
# 실행부 (원래 코드와 동일한 인터페이스 유지)
# ---------------------------------------------------------------------
samples = run_chatbot_on_testset(testset_df)

result_df = run_ragas_evaluation(samples)
result_df.to_csv(RESULT_CSV, index=False, encoding="utf-8-sig")

metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

print("===== 평가 결과 (평균) =====")
for col in metric_cols:
    print(f"{col}: {result_df[col].mean():.4f}")

result_df

C:\Users\박동관\AppData\Local\Temp\ipykernel_17020\1755738270.py:5: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
C:\Users\박동관\AppData\Local\Temp\ipykernel_17020\1755738270.py:5: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
C:\Users\박동관\AppData\Local\Temp\ipykernel_17020\1755738270.py:5: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.

[1/20] OK  병역법에 따라 병역준비역으로 편입된 사람에게 어떤 절차나 사항을 통지해야...
[2/20] OK  군 복무에 관련된 위원회의 회의는 어떻게 운영되나요?...
[3/20] OK  국군 회원과 기업 회원이 받을 수 있는 혜택의 제공 주기는 어떻게 되나요...
[4/20] OK  전역할 때 전역증은 어떻게 받을 수 있나요?...
[5/20] OK  군 복무 중 성고충 문제에 대한 전문 상담 제도는 어떤 항목에 포함되어 ...
[6/20] OK  2026년에는 초급간부의 기본급이 얼마나 인상되나요?...


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.vector.queryNodes('article_vector_index', $top_k, $query_embedding)\nYIELD node, score\nRETURN node.id AS id, node.name AS name, node.description AS description, score\nORDER BY score DESC"


[7/20] OK  전직지원교육을 위탁할 수 있는 기관은 어떤 기준으로 선정되나요?...
[8/20] OK  군 복무 중 병 복지 혜택에는 어떤 것들이 있나요?...
[9/20] OK  박사과정전문연구요원이 박사학위 취득 유예기간을 신청하려면 어떤 절차를 따...
[10/20] OK  2026년에 소위 계급의 급여는 얼마나 되나요?...


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.vector.queryNodes('article_vector_index', $top_k, $query_embedding)\nYIELD node, score\nRETURN node.id AS id, node.name AS name, node.description AS description, score\nORDER BY score DESC"


...

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.vector.queryNodes('article_vector_index', $top_k, $query_embedding)\nYIELD node, score\nRETURN node.id AS id, node.name AS name, node.description AS description, score\nORDER BY score DESC"


[20/20] OK  군 복무 중인 사람에게 연금 분할이 어떻게 적용되나요?...

수집 완료: 20/20 문항


C:\Users\박동관\AppData\Local\Temp\ipykernel_17020\1755738270.py:21: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model=EVAL_LLM_MODEL, temperature=0))
C:\Users\박동관\AppData\Local\Temp\ipykernel_17020\1755738270.py:22: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())  # answer_relevancy가 임베딩 필요
Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 

===== 평가 결과 (평균) =====
faithfulness: 0.7803
answer_relevancy: 0.5348
context_precision: 0.8000
context_recall: 0.6500


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,병역법에 따라 병역준비역으로 편입된 사람에게 어떤 절차나 사항을 통지해야 하나요?,[--- [참조 조문 1] ---\n[조문] 병역변경 등의 통지 (ID: 병역법::...,"병역준비역으로 편입되면, 지방병무청장이 그 사람한테 **“네가 병역준비역으로 편입됐...","병역법에 따르면 지방병무청장은 병역준비역으로 편입된 사람에게 현역, 보충역, 대체역...",1.000000,0.804714,1.0,1.0
1,군 복무에 관련된 위원회의 회의는 어떻게 운영되나요?,[--- [참조 문서 1] ---\n10 \n2026 초급간부 길라잡이\n-①-Ⅰ\...,"야, 그 질문은 **위원회 종류에 따라 완전 달라**. \n지금 검색된 자료는 *...","위원회의 회의는 재적위원 과반수의 출석으로 개의하고, 출석위원 3분의 2 이상의 찬...",0.666667,0.000000,1.0,0.0
2,국군 회원과 기업 회원이 받을 수 있는 혜택의 제공 주기는 어떻게 되나요?,[--- [참조 문서 1] ---\n이 표는 국군/기업 회원 대상 숙박·예약 혜택을...,"국군 회원이랑 기업 회원 혜택은 **주기가 딱 정해진 항목이 따로 있고, 기업 조건...","국군 회원은 전용 쿠폰을 월 1회 받을 수 있으며, 기업 회원은 쿠폰을 월별로 변동...",1.000000,0.000000,1.0,1.0
3,전역할 때 전역증은 어떻게 받을 수 있나요?,[--- [참조 문서 1] ---\nPart 02 자기개발 분야\n32 2026 병...,"어이구, 전역증은 전역한다고 바로 손에 쥐어주는 게 아니라 절차가 있지.\n\n정리...",전역증은 전역 당일 전역신고 후 개인에게 지급됩니다. 만약 나라사랑카드를 보유하고 ...,0.923077,0.733308,1.0,1.0
4,군 복무 중 성고충 문제에 대한 전문 상담 제도는 어떤 항목에 포함되어 있나요?,[--- [참조 문서 1] ---\n2026 병 복지 길라잡이 47\n성고충전문상담...,성고충 전문상담 제도는 주로 이런 항목을 포함하오:\n\n- **성적 괴롭힘**\n...,성고충 문제에 대한 전문 상담 제도는 '성고충전문상담관 운영' 항목에 포함되어 있습...,1.000000,0.762534,1.0,0.5
5,2026년에는 초급간부의 기본급이 얼마나 인상되나요?,[--- [참조 문서 1] ---\n6 \n 기본급 및 각종 수당 인상 \n \n•...,2026년에는 **초급간부 기본급이 6.6% 인상**되오. \n대상은 **하사·중...,"2026년에는 초급간부의 기본급이 6.6% 인상됩니다. 예를 들어, 소위는 2,15...",0.916667,0.841790,1.0,1.0
6,전직지원교육을 위탁할 수 있는 기관은 어떤 기준으로 선정되나요?,[--- [참조 조문 1] ---\n[조문] 전직지원교육 대상 등 (ID: 군인사법...,"전직지원교육을 **위탁할 기관이나 단체**는 아무 데나 맡기는 게 아니고, **전직...",전직지원교육 관련 업무는 국방부장관이 대통령령으로 정한 바에 따라 위탁할 수 있습니...,1.000000,0.844517,1.0,1.0
7,군 복무 중 병 복지 혜택에는 어떤 것들이 있나요?,[--- [참조 문서 1] ---\n보수\n급식 및 물자\n병영생활관 개선\n현역병...,군 복무 중 병 복지 혜택은 꽤 여러 갈래로 나뉘어 있지. 자료 기준으로 보면 대표...,"병 복지 혜택에는 급여, 휴가, 의료 서비스, 주거 지원 등이 포함됩니다. 이러한 ...",0.833333,0.819146,1.0,0.0
8,박사과정전문연구요원이 박사학위 취득 유예기간을 신청하려면 어떤 절차를 따라야 하나요?,[--- [참조 문서 1] ---\n56 \n2026 초급간부 길라잡이\n-①-Ⅰ\...,이 질문은 **박사과정전문연구요원의 박사학위 취득 유예기간 신청 절차**를 물으셨는...,박사과정전문연구요원이 박사학위 취득 유예기간을 신청하려면 복무하고 있는 병역지정업체...,0.125000,0.000000,0.0,0.0
9,2026년에 소위 계급의 급여는 얼마나 되나요?,[--- [참조 문서 1] ---\n6 \n 기본급 및 각종 수당 인상 \n \n•...,"2026년 **소위 기본급은 2,150,400원**이오. \n자료 기준으로는 20...","2026년에 소위 계급의 급여는 2,150,400원입니다.",0.800000,0.000000,1.0,1.0


## 4. 평가 (자체 구현, RAGAS 미사용)

RAGAS `evaluate` 대신, `ChatOpenAI`를 LLM 심사위원으로 삼아 4개 지표를 직접 계산한다.
모든 프롬프트와 채점 근거(reason)를 한국어로 작성했기 때문에 결과도 한국어로 나온다.

- **Faithfulness** — 답변이 검색된 컨텍스트에 충실한가 (환각 여부)
- **AnswerRelevancy** — 답변이 질문에 적절한가
- **ContextPrecision** — 검색된 컨텍스트 중 정답(reference)과 실제로 관련된 비율이 높은가
- **ContextRecall** — 정답(reference)에 필요한 내용을 컨텍스트가 충분히 포함하는가

각 지표는 0.0 ~ 1.0 사이 점수와 한국어 근거(reason)를 함께 반환한다.


In [41]:
# =====================================================================
# LLM 심사위원 프롬프트 (전부 한국어, JSON 출력 강제)
# =====================================================================
FAITHFULNESS_PROMPT = """당신은 RAG 챗봇의 답변이 검색된 컨텍스트에 충실한지 평가하는 심사위원이다.

[질문]
{question}

[검색된 컨텍스트]
{contexts}

[챗봇 답변]
{answer}

위 답변에 포함된 주장들이 컨텍스트만으로 뒷받침되는지 평가해라.
컨텍스트에 없는 내용을 답변이 지어냈다면(환각) 점수를 낮게 준다.

0.0(전혀 근거 없음) ~ 1.0(완전히 근거함) 사이의 점수를 매기고,
다른 설명 없이 반드시 아래 JSON 형식으로만 한국어로 응답해라.

{{"score": 0.0에서 1.0 사이 숫자, "reason": "한국어로 작성한 간단한 근거"}}
"""

ANSWER_RELEVANCY_PROMPT = """당신은 RAG 챗봇의 답변이 질문에 얼마나 적절한지 평가하는 심사위원이다.

[질문]
{question}

[챗봇 답변]
{answer}

답변이 질문의 의도에 맞게, 불필요한 내용 없이 직접적으로 답하고 있는지 평가해라.
질문과 무관하거나 동문서답이면 점수를 낮게 준다.

0.0(전혀 관련 없음) ~ 1.0(매우 적절함) 사이의 점수를 매기고,
다른 설명 없이 반드시 아래 JSON 형식으로만 한국어로 응답해라.

{{"score": 0.0에서 1.0 사이 숫자, "reason": "한국어로 작성한 간단한 근거"}}
"""

CONTEXT_PRECISION_PROMPT = """당신은 RAG 검색 결과의 정밀도를 평가하는 심사위원이다.

[질문]
{question}

[정답(reference)]
{reference}

[검색된 컨텍스트 목록]
{contexts}

검색된 컨텍스트 중 실제로 정답을 뒷받침하는 데 도움이 되는 컨텍스트의 비율이
얼마나 높은지 평가해라. 정답과 무관한 컨텍스트가 많이 섞여 있으면 점수를 낮게 준다.

0.0(대부분 무관함) ~ 1.0(전부 관련 있음) 사이의 점수를 매기고,
다른 설명 없이 반드시 아래 JSON 형식으로만 한국어로 응답해라.

{{"score": 0.0에서 1.0 사이 숫자, "reason": "한국어로 작성한 간단한 근거"}}
"""

CONTEXT_RECALL_PROMPT = """당신은 RAG 검색 결과의 재현율(recall)을 평가하는 심사위원이다.

[정답(reference)]
{reference}

[검색된 컨텍스트 목록]
{contexts}

정답을 구성하는 데 필요한 정보가 검색된 컨텍스트에 충분히 포함되어 있는지 평가해라.
정답의 핵심 내용 중 컨텍스트에 빠진 부분이 많으면 점수를 낮게 준다.

0.0(전혀 포함 안 됨) ~ 1.0(완전히 포함됨) 사이의 점수를 매기고,
다른 설명 없이 반드시 아래 JSON 형식으로만 한국어로 응답해라.

{{"score": 0.0에서 1.0 사이 숫자, "reason": "한국어로 작성한 간단한 근거"}}
"""


def _judge(llm, prompt: str) -> tuple[float, str]:
    """LLM을 한 번 호출해 (score, reason)을 파싱해 반환한다. 실패 시 0점 처리."""
    try:
        resp = llm.invoke([{"role": "user", "content": prompt}])
        parsed = _parse_json_response(resp.content)
        score = float(parsed.get("score", 0.0))
        score = max(0.0, min(1.0, score))
        reason = str(parsed.get("reason", ""))
        return score, reason
    except Exception as e:
        return 0.0, f"평가 실패: {e}"


def eval_faithfulness(llm, question, answer, contexts) -> tuple[float, str]:
    ctx_text = "\n\n".join(f"- {c}" for c in contexts) or "(검색된 컨텍스트 없음)"
    prompt = FAITHFULNESS_PROMPT.format(question=question, contexts=ctx_text, answer=answer)
    return _judge(llm, prompt)


def eval_answer_relevancy(llm, question, answer) -> tuple[float, str]:
    prompt = ANSWER_RELEVANCY_PROMPT.format(question=question, answer=answer)
    return _judge(llm, prompt)


def eval_context_precision(llm, question, reference, contexts) -> tuple[float, str]:
    ctx_text = "\n\n".join(f"{idx + 1}. {c}" for idx, c in enumerate(contexts)) or "(검색된 컨텍스트 없음)"
    prompt = CONTEXT_PRECISION_PROMPT.format(
        question=question, reference=reference or "(정답 없음)", contexts=ctx_text
    )
    return _judge(llm, prompt)


def eval_context_recall(llm, reference, contexts) -> tuple[float, str]:
    ctx_text = "\n\n".join(f"- {c}" for c in contexts) or "(검색된 컨텍스트 없음)"
    prompt = CONTEXT_RECALL_PROMPT.format(reference=reference or "(정답 없음)", contexts=ctx_text)
    return _judge(llm, prompt)


def run_evaluation(samples: list[EvalSample]) -> pd.DataFrame:
    """RAGAS 없이 LLM 심사위원 방식으로 4개 지표를 한국어로 평가한다."""
    evaluator_llm = ChatOpenAI(model=EVAL_LLM_MODEL, temperature=0)

    rows = []
    total = len(samples)
    for i, s in enumerate(samples, start=1):
        f_score, f_reason = eval_faithfulness(evaluator_llm, s.user_input, s.response, s.retrieved_contexts)
        ar_score, ar_reason = eval_answer_relevancy(evaluator_llm, s.user_input, s.response)
        cp_score, cp_reason = eval_context_precision(evaluator_llm, s.user_input, s.reference, s.retrieved_contexts)
        cr_score, cr_reason = eval_context_recall(evaluator_llm, s.reference, s.retrieved_contexts)

        rows.append(
            {
                "user_input": s.user_input,
                "response": s.response,
                "reference": s.reference,
                "faithfulness": f_score,
                "faithfulness_reason": f_reason,
                "answer_relevancy": ar_score,
                "answer_relevancy_reason": ar_reason,
                "context_precision": cp_score,
                "context_precision_reason": cp_reason,
                "context_recall": cr_score,
                "context_recall_reason": cr_reason,
            }
        )
        print(
            f"[{i}/{total}] 평가 완료 "
            f"(faithfulness={f_score:.2f}, relevancy={ar_score:.2f}, "
            f"precision={cp_score:.2f}, recall={cr_score:.2f})"
        )

    return pd.DataFrame(rows)


In [42]:
result_df = run_evaluation(samples)
result_df.to_csv(RESULT_CSV, index=False, encoding="utf-8-sig")

metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

print("===== 평가 결과 (평균) =====")
for col in metric_cols:
    print(f"{col}: {result_df[col].mean():.4f}")

result_df


[1/20] 평가 완료 (faithfulness=1.00, relevancy=1.00, precision=1.00, recall=1.00)
[2/20] 평가 완료 (faithfulness=1.00, relevancy=1.00, precision=0.80, recall=0.50)
[3/20] 평가 완료 (faithfulness=1.00, relevancy=0.80, precision=0.80, recall=1.00)
[4/20] 평가 완료 (faithfulness=1.00, relevancy=1.00, precision=0.80, recall=0.80)
[5/20] 평가 완료 (faithfulness=1.00, relevancy=1.00, precision=0.80, recall=1.00)
[6/20] 평가 완료 (faithfulness=1.00, relevancy=1.00, precision=0.80, recall=1.00)
[7/20] 평가 완료 (faithfulness=1.00, relevancy=1.00, precision=0.20, recall=0.00)
[8/20] 평가 완료 (faithfulness=1.00, relevancy=1.00, precision=0.60, recall=0.30)
[9/20] 평가 완료 (faithfulness=1.00, relevancy=1.00, precision=1.00, recall=0.80)
[10/20] 평가 완료 (faithfulness=1.00, relevancy=1.00, precision=0.90, recall=1.00)
[11/20] 평가 완료 (faithfulness=1.00, relevancy=1.00, precision=0.90, recall=1.00)
[12/20] 평가 완료 (faithfulness=1.00, relevancy=1.00, precision=0.10, recall=0.00)
[13/20] 평가 완료 (faithfulness=1.00, relevancy=1.00, precision=0

,user_input,response,reference,faithfulness,faithfulness_reason,answer_relevancy,answer_relevancy_reason,context_precision,context_precision_reason,context_recall,context_recall_reason
0,병역법에 따라 병역준비역으로 편입된 사람에게 어떤 절차나 사항을 통지해야 하나요?,병역법에 따라 병역준비역으로 편입된 사람에게는 지방병무청장이 편입 사실과 병역 이행...,"병역법에 따르면 지방병무청장은 병역준비역으로 편입된 사람에게 현역, 보충역, 대체역...",1.0,"챗봇의 답변은 검색된 컨텍스트에 충실하며, 병역준비역으로 편입된 사람에게 통지해야 ...",1.0,질문에 대한 구체적인 절차와 내용을 명확히 설명하고 있어 매우 적절하다.,1.0,검색된 컨텍스트가 병역법에 따른 병역준비역 편입 절차와 통지 사항에 대한 내용을 명...,1.0,검색된 컨텍스트에서 병역법에 따른 지방병무청장의 통지 의무와 관련된 모든 핵심 내용...
1,군 복무에 관련된 위원회의 회의는 어떻게 운영되나요?,군 복무에 관련된 위원회의 회의 운영은 다음과 같은 절차와 규정에 따라 이루어집니다...,"위원회의 회의는 재적위원 과반수의 출석으로 개의하고, 출석위원 3분의 2 이상의 찬...",1.0,답변은 군 복무 관련 위원회의 회의 운영에 대한 절차와 규정을 정확히 반영하고 있으...,1.0,질문에 대한 구체적인 절차와 규정을 명확하게 설명하고 있어 매우 적절함.,0.8,검색된 컨텍스트 중 일부는 위원회의 운영 방식에 대한 정보를 포함하고 있어 정답을 ...,0.5,"위원회의 회의 개의 및 의결에 관한 정보는 일부 포함되어 있으나, 출석위원 3분의 ..."
2,국군 회원과 기업 회원이 받을 수 있는 혜택의 제공 주기는 어떻게 되나요?,국군 회원과 기업 회원이 받을 수 있는 혜택의 제공 주기는 다음과 같습니다:\n\n...,"국군 회원은 전용 쿠폰을 월 1회 받을 수 있으며, 기업 회원은 쿠폰을 월별로 변동...",1.0,"답변은 검색된 컨텍스트에 명시된 혜택 제공 주기를 정확히 반영하고 있으며, 추가적인...",0.8,"국군 회원과 기업 회원의 혜택 제공 주기를 구체적으로 설명하였으나, 'Gold 특가...",0.8,검색된 컨텍스트 중 일부는 국군 및 기업 회원의 혜택에 대한 정보를 포함하고 있어 ...,1.0,검색된 컨텍스트에 국군 회원과 기업 회원의 쿠폰 제공 주기 및 혜택이 명확하게 포함...
3,전역할 때 전역증은 어떻게 받을 수 있나요?,전역할 때 전역증을 받는 절차는 다음과 같습니다:\n\n1. **소속 부대장에게 요...,전역증은 전역 당일 전역신고 후 개인에게 지급됩니다. 만약 나라사랑카드를 보유하고 ...,1.0,"전역증을 소속 부대장에게 요청하여 받는 절차와 전역증의 내용, 분실 시 재교부 신청...",1.0,전역증을 받는 절차를 구체적으로 설명하고 있어 질문의 의도에 매우 적절하다.,0.8,"전역증 지급 절차에 대한 정보가 포함된 조문이 있어 관련성이 높지만, 다른 조문들은...",0.8,"전역증의 지급 절차와 관련된 정보는 포함되어 있으나, 나라사랑카드와 이메일 발송에 ..."
4,군 복무 중 성고충 문제에 대한 전문 상담 제도는 어떤 항목에 포함되어 있나요?,군 복무 중 성고충 문제에 대한 전문 상담 제도는 다음과 같은 항목에 포함되어 있습...,성고충 문제에 대한 전문 상담 제도는 '성고충전문상담관 운영' 항목에 포함되어 있습...,1.0,"답변은 검색된 컨텍스트에 포함된 성고충전문상담관 운영 및 상담 내용, 상담 신청 방...",1.0,"질문에 대한 구체적인 답변을 제공하며, 관련된 항목들을 명확하게 설명하고 있어 매우...",0.8,검색된 컨텍스트 중 성고충전문상담관 운영에 대한 정보가 포함되어 있어 정답을 뒷받침...,1.0,검색된 컨텍스트에 성고충전문상담관 운영에 대한 정보가 충분히 포함되어 있어 정답의 ...
5,2026년에는 초급간부의 기본급이 얼마나 인상되나요?,2026년에는 초급간부의 기본급이 6.6% 인상됩니다. 구체적인 인상 금액은 다음과...,"2026년에는 초급간부의 기본급이 6.6% 인상됩니다. 예를 들어, 소위는 2,15...",1.0,챗봇의 답변은 검색된 컨텍스트에 명시된 초급간부의 기본급 인상률과 구체적인 금액을 ...,1.0,질문에 대한 구체적인 인상 금액과 비율을 명확하게 제시하여 매우 적절한 답변이다.,0.8,검색된 컨텍스트 중 일부는 초급간부의 기본급 인상에 대한 정보를 포함하고 있어 관련...,1.0,검색된 컨텍스트 중 '기본급 6.6% 인상'과 각 계급별 인상 금액이 명확하게 포함...
6,전직지원교육을 위탁할 수 있는 기관은 어떤 기준으로 선정되나요?,전직지원교육을 위탁할 수 있는 기관은 다음과 같은 기준에 따라 선정됩니다:\n\n1...,전직지원교육 관련 업무는 국방부장관이 대통령령으로 정한 바에 따라 위탁할 수 있습니...,1.0,답변은 군인사법 시행령 제60조의2에 명시된 전직지원교육을 위탁할 수 있는 기관의 ...,1.0,"질문에 대한 명확한 기준을 제시하며, 관련 법령을 근거로 설명하고 있어 매우 적절하다.",0.2,"검색된 컨텍스트는 전직지원교육에 대한 직접적인 정보가 부족하고, 대부분 병역 관련 ...",0.0,검색된 컨텍스트에는 전직지원교육 관련 업무를 국방부장관이 위탁할 수 있다는 내용이 ...
7,군 복무 중 병 복지 혜택에는 어떤 것들이 있나요?,군 복무 중 병사들이 받을 수 있는 복지 혜택은 다양합니다. 주요 혜택은 다음과 같...,"병 복지 혜택에는 급여, 휴가, 의료 서비스, 주거 지원 등이 포함됩니다. 이러한 ...",1.0,"답변에 포함된 모든 혜택은 검색된 컨텍스트에서 명시적으로 언급되었으며, 군 복무 중...",1.0,군 복무 중 병사들이 받을 수 있는 다양한 복지 혜택을 구체적으로 나열하여 질문에 ...,0.6,"검색된 컨텍스트 중 일부는 병 복지 혜택과 관련된 정보를 포함하고 있지만, 많은 부...",0.3,"검색된 컨텍스트에는 병 복지 혜택에 대한 구체적인 내용이 부족하고, 급여, 휴가, ..."
8,박사과정전문연구요원이 박사학위 취득 유예기간을 신청하려면 어떤 절차를 따라야 하나요?,박사과정전문연구요원이 박사학위 취득 유예기간을 신청하려면 다음과 같은 절차를 따라야...,박사과정전문연구요원이 박사학위 취득 유예기간을 신청하려면 복무하고 있는 병역지정업체...,1.0,"답변은 검색된 컨텍스트에 명시된 병역법 시행령 제78조의4에 충실하며, 유예기간 신...",1.0,질문에 대한 절차를 명확하고 구체적으로 설명하고 있어 매우 적절하다.,1.0,검색된 컨텍스트가 박사과정전문연구요원의 박사학위 취득 유예기간 신청 절차에 대한 내...,0.8,검색된 컨텍스트에서 박사학위 취득 유예기간 신청 절차와 관련된 정보가 포함되어 있으...
9,2026년에 소위 계급의 급여는 얼마나 되나요?,"2026년 소위 계급의 기본급은 2,150,400원으로 인상됩니다. 이는 2025년...","2026년에 소위 계급의 급여는 2,150,400원입니다.",1.0,"2026년 소위 계급의 기본급이 2,150,400원으로 인상된다는 내용이 검색된 컨...",1.0,"질문에 대한 정확한 급여 정보를 제공하며, 추가적인 수당에 대한 설명도 포함되어 있...",0.9,검색된 컨텍스트 중에서 소위 계급의 급여에 대한 정보가 포함된 문서가 명확하게 존재...,1.0,"검색된 컨텍스트 중 '2026년 초급간부 기본급'에 대한 정보가 포함되어 있어, 정..."
